# **Fairness-Aware Age Estimation: Bias Detection and Mitigation**
UB Master in Fundamental Principels of Data Science (2025-2026)


## Checking the pytorch version
 - This notebook was successfully tested on version = 2.9.0+cu126

In [1]:
import torch
print(torch.__version__)

2.10.0


## Importing required libraries

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import csv
from PIL import Image
from timm import create_model
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from tqdm import tqdm

# **Downloading the Appa-Real Dataset**
- Please, check the **detailed instructions in notebook Part I**.

# **Mount Google Drive to save the trained model on the cloud**

In [3]:
from google.colab import drive
drive.mount('/content/drive')
# Note, the default path will be: '/content/gdrive/MyDrive/'
# In my case, the final path will be: '/content/gdrive/MyDrive/temp/' as I
# created a '/temp/' folder in my google drive for this purpose.

In [ ]:
!cp "/content/drive/MyDrive/appa-real-dataset_v2.zip" /content/
!cp "/content/drive/MyDrive/UTKFace.zip" /content/
!unzip appa-real-dataset_v2.zip -d /content/
!unzip UTKFace.zip -d /content/

In [ ]:
!cp "/content/drive/MyDrive/utk_filtered.csv" /content/

# **Defining the Data Loader Class**
- In this example, metadata information is loaded but not used. Future implementations can take benefit of it.
- Note that age labels are divided by 100 (assuming 100 is the max age found in the dataset) so that the age values can be normalized to be in the range of 0 and 1. This way, we can add a sigmoid activation in the last layer of our model.
- The method `__getitem__` implements a **simple data augmentation strategy based only on the age label**: all training samples with age > 60 are transformed using the transformations defined in `augmented_transforms`. All other training samples are transformed using the same transformations, but with a probability of 25%.


In [4]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image


class AgeEstimationDataset(Dataset):
    def __init__(
        self,
        appa_dir,
        utk_dir,
        csv_file,
        transforms=None,
        age_normalization_factor=100.0,
    ):
        self.appa_dir   = appa_dir
        self.utk_dir    = utk_dir
        self.transforms = transforms
        self.data_info  = pd.read_csv(csv_file)
        self.age_norm   = age_normalization_factor

        self.gender_map = {"male": 0, "female": 1}
        self.eth_map    = {
            "asian": 0, "afroamerican": 1,
            "caucasian": 2, "other": 3,
        }
        self.exp_map = {
            "neutral": 0, "slightlyhappy": 1,
            "happy": 2, "other": 3,
            "unknown": 4,   # sentinel — masked out in loss
        }

    def __len__(self):
        return len(self.data_info)

    def _resolve_image_path(self, row):
        source   = row["source"]
        raw_file = row["imageid"]
        if source == "appa":
            image_id = f"{int(float(raw_file)):06d}.jpg"
            return os.path.join(self.appa_dir, image_id)
        elif source == "utk":
            return os.path.join(self.utk_dir, str(raw_file))
        raise ValueError(f"Unknown dataset source: {source}")

    def __getitem__(self, idx):
        row = self.data_info.iloc[idx]

        # ── Image ────────────────────────────────────────────
        image_path = self._resolve_image_path(row)
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error loading image: {image_path}") from e

        if self.transforms is not None:
            image = self.transforms(image)

        # ── Labels ───────────────────────────────────────────
        age = torch.tensor(
            float(row["age"]) / self.age_norm, dtype=torch.float32
        )

        gender = self.gender_map.get(row["gender"], 0)

        ethnicity = self.eth_map.get(row["ethnicity"], 3)

        # UTK carries no expression → sentinel 4, masked in loss
        exp_str = "unknown" if row["source"] == "utk" else row["expression"]
        expression = self.exp_map.get(exp_str, 4)

        labels = {
            "age":        age,
            "gender":     torch.tensor(gender,     dtype=torch.long),
            "ethnicity":  torch.tensor(ethnicity,  dtype=torch.long),
            "expression": torch.tensor(expression, dtype=torch.long),
        }

        return image, labels

In [ ]:
# --- Process APPA Train ---
df_appa = pd.read_csv("/content/appa-real-dataset_v2/labels_metadata_train.csv")
df_appa.columns = ['imageid', 'age', 'gender', 'ethnicity', 'expression']
df_appa['source'] = 'appa'

# --- Process UTK (Filtered) ---
# Assuming you've already generated utk_filtered.csv with standard names
df_utk = pd.read_csv("/content/utk_filtered.csv")
df_utk.columns = ['imageid', 'age', 'gender', 'ethnicity', 'expression']
df_utk['source'] = 'utk'

# --- Combine ---
df_train_combined = pd.concat([df_appa, df_utk], ignore_index=True)
df_train_combined.to_csv("train_final.csv", index=False)

# --- Process APPA Valid ---
df_valid = pd.read_csv("/content/appa-real-dataset_v2/labels_metadata_valid.csv")
df_valid.columns = ['imageid', 'age', 'gender', 'ethnicity', 'expression']
df_valid['source'] = 'appa'
df_valid.to_csv("valid_final.csv", index=False)

# **Defining the base image transformations**

In [5]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# **Loading the Train and Validation sets**

In [7]:
# Create dataset and dataloader (train set):

APPA_TRAIN = "/content/appa-real-dataset_v2/train_data"
APPA_VALID = "/content/appa-real-dataset_v2/valid_data"
UTK_PATH = "/content/UTKFace"


# TRAINING SET
dataset_train = AgeEstimationDataset(
    appa_dir=APPA_TRAIN,
    utk_dir=UTK_PATH,
    csv_file="train_final.csv",
    transforms=base_transforms
)

dataloader_train = DataLoader(
    dataset_train,
    batch_size=64,
    num_workers=2,
    pin_memory=True,
    shuffle=True
)

# VALIDATION SET (APPA Only)
dataset_valid = AgeEstimationDataset(
    appa_dir=APPA_VALID,
    utk_dir="", # Not used
    csv_file="valid_final.csv",
    transforms=base_transforms
)

dataloader_valid = DataLoader(dataset_valid, batch_size=64, shuffle=False)

Total number of train samples: 4065
Total number of valid samples: 1482


# **Loading the pretrained ViT baselone model and adapting it to our problem**

- ViT normally outputs class scores for classification tasks; here, we adapt it for regression by setting **num_classes=1**.

> **Note:** This notebook is intended as a starting point. For your deliverables, avoid making only minor modifications. Instead, explore your creativity and try more substantial improvements.

- **It is also recommended to use the same architecture when comparing results with and without data augmentation, in order to ensure a fair comparison.**


In [10]:
import torch
import torch.nn as nn
from timm import create_model


class AgeEstimationConvnext(nn.Module):
    def __init__(self):
        super().__init__()

        # ── Backbone ─────────────────────────────────────────
        self.backbone = create_model(
            "convnext_tiny",
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        backbone_dim = self.backbone.num_features  # 768

        # ── Shared trunk ─────────────────────────────────────
        self.shared_trunk = nn.Sequential(
            nn.Linear(backbone_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.LayerNorm(512),
        )

        # ── Main task: age regression ─────────────────────────
        self.age_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

        # ── Auxiliary tasks ───────────────────────────────────
        self.gender_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.GELU(),
            nn.Linear(64, 2),
        )
        self.ethnicity_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.GELU(),
            nn.Linear(64, 4),
        )
        self.expression_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.GELU(),
            nn.Linear(64, 4),   # "unknown" (idx 4) masked in loss
        )

    def forward(self, x):
        shared = self.shared_trunk(self.backbone(x))

        return {
            "age":        self.age_head(shared).squeeze(1),  # (B,)
            "gender":     self.gender_head(shared),           # (B, 2)
            "ethnicity":  self.ethnicity_head(shared),        # (B, 4)
            "expression": self.expression_head(shared),       # (B, 4)
        }

In [11]:
# creating the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeEstimationConvnext().to(device)

# Defining an auxiliary function to plot the training history

In [12]:
# Function to plot training curves
def plot_training_curves(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

# **Defining the Training function**
- Our training code includes **early stopping** and **automatic saving of the best model**. Early stopping monitors the validation loss during training and halts training if the model stops improving for a specified number of epochs, helping to prevent overfitting and save computational resources. At the same time, the model with the lowest validation loss is automatically saved, ensuring that we keep the best-performing version for evaluation or deployment.

In [13]:
import copy
import torch
import torch.nn.functional as F
from tqdm import tqdm


def train_model(
    model,
    dataloaders,
    criterion,           # age loss (nn.L1Loss recommended)
    optimizer,
    scheduler,
    num_epochs,
    patience,
    model_path,
    device,
    w_age:        float = 1.0,
    w_gender:     float = 0.3,
    w_ethnicity:  float = 0.3,
    w_expression: float = 0.2,
):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss      = float("inf")
    train_losses   = []
    val_losses     = []
    early_counter  = 0
    scaler         = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running = {
                "total": 0.0, "age": 0.0,
                "gender": 0.0, "ethnicity": 0.0, "expression": 0.0
            }

            with tqdm(dataloaders[phase], desc=phase.upper()) as t:
                for images, labels in t:
                    images     = images.to(device)
                    age_tgt    = labels["age"].to(device)
                    gen_tgt    = labels["gender"].to(device)
                    eth_tgt    = labels["ethnicity"].to(device)
                    exp_tgt    = labels["expression"].to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == "train"):
                        with torch.cuda.amp.autocast():
                            out = model(images)   # no meta argument

                            loss_age = criterion(out["age"], age_tgt)
                            loss_gen = F.cross_entropy(
                                out["gender"], gen_tgt)
                            loss_eth = F.cross_entropy(
                                out["ethnicity"], eth_tgt)

                            # Mask UTK samples (expression == 4 → unknown)
                            exp_mask = exp_tgt < 4
                            loss_exp = (
                                F.cross_entropy(
                                    out["expression"][exp_mask],
                                    exp_tgt[exp_mask],
                                )
                                if exp_mask.any()
                                else torch.tensor(0.0, device=device)
                            )

                            loss = (
                                w_age        * loss_age
                              + w_gender     * loss_gen
                              + w_ethnicity  * loss_eth
                              + w_expression * loss_exp
                            )

                        if phase == "train":
                            scaler.scale(loss).backward()
                            torch.nn.utils.clip_grad_norm_(
                                model.parameters(), 1.0)
                            scaler.step(optimizer)
                            scaler.update()

                    n = images.size(0)
                    running["total"]      += loss.item()      * n
                    running["age"]        += loss_age.item()  * n
                    running["gender"]     += loss_gen.item()  * n
                    running["ethnicity"]  += loss_eth.item()  * n
                    running["expression"] += loss_exp.item()  * n

                    t.set_postfix(
                        total = f"{loss.item():.4f}",
                        age   = f"{loss_age.item():.4f}",
                        gen   = f"{loss_gen.item():.4f}",
                        eth   = f"{loss_eth.item():.4f}",
                        exp   = f"{loss_exp.item():.4f}",
                    )

            n_samples = len(dataloaders[phase].dataset)
            e = {k: v / n_samples for k, v in running.items()}

            print(
                f"{phase.upper():5s} | "
                f"total={e['total']:.4f}  "
                f"age={e['age']:.4f}  "
                f"gender={e['gender']:.4f}  "
                f"ethnicity={e['ethnicity']:.4f}  "
                f"expression={e['expression']:.4f}"
            )

            if phase == "train":
                train_losses.append(e["total"])
            else:
                val_losses.append(e["total"])
                scheduler.step(e["total"])

                if e["total"] < best_loss:
                    best_loss      = e["total"]
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(best_model_wts, model_path)
                    print("  ✔ Saved best model")
                    early_counter = 0
                else:
                    early_counter += 1

                if early_counter >= patience:
                    print("\nEarly stopping triggered")
                    model.load_state_dict(best_model_wts)
                    return model, train_losses, val_losses

    model.load_state_dict(best_model_wts)
    return model, train_losses, val_losses

In [14]:
def set_backbone_freeze(model, freeze=True):
    """
    Freezes or unfreezes the ViT backbone.
    """
    for param in model.backbone.parameters():
        param.requires_grad = not freeze
    
    status = "Frozen" if freeze else "Unfrozen"
    print(f"Backbone status: {status}")

In [15]:
# Function to plot training curves
def plot_training_curves(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

# **Training the Model (now with data augmentation)**

- In the next cell, **we train our model using the same architecture and hyperparameters as in Part I**.
- The only difference is that **we now apply data augmentation**, as described above.



In [16]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_TRAIN    = True
model_filename = "./best_multitask_model.pth"
dataloaders    = {"train": dataloader_train, "val": dataloader_valid}

criterion = nn.L1Loss()

AUX_WEIGHTS = dict(
    w_age        = 1.0,
    w_gender     = 0.3,
    w_ethnicity  = 0.3,
    w_expression = 0.2,
)

def set_backbone_freeze(model, freeze: bool):
    for p in model.backbone.parameters():
        p.requires_grad = not freeze


if MODEL_TRAIN:
    # ---------------------------------------------------------
    # STAGE 1: Warm-up — heads + trunk only, backbone frozen
    # ---------------------------------------------------------
    print("\n>>> STAGE 1: Warming up heads (backbone frozen)")
    set_backbone_freeze(model, freeze=True)

    optimizer_v1 = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr           = 3e-4,
        weight_decay = 1e-2,
    )
    scheduler_v1 = ReduceLROnPlateau(
        optimizer_v1, mode="min", factor=0.5, patience=3, min_lr=1e-6
    )

    model, hist1_train, hist1_val = train_model(
        model,
        dataloaders,
        criterion  = criterion,
        optimizer  = optimizer_v1,
        scheduler  = scheduler_v1,
        num_epochs = 10,
        patience   = 4,
        model_path = model_filename,
        device     = device,
        **AUX_WEIGHTS,
    )

    # ---------------------------------------------------------
    # STAGE 2: Fine-tuning — full model, layer-wise LR decay
    # ---------------------------------------------------------
    print("\n>>> STAGE 2: Fine-tuning full model")
    set_backbone_freeze(model, freeze=False)

    optimizer_v2 = AdamW(
        [
            # Backbone — pretrained, use very small LR
            {"params": model.backbone.parameters(),        "lr": 5e-6},
            # Shared trunk — moderate LR
            {"params": model.shared_trunk.parameters(),    "lr": 2e-4},
            # All heads — highest LR
            {"params": model.age_head.parameters(),        "lr": 2e-4},
            {"params": model.gender_head.parameters(),     "lr": 2e-4},
            {"params": model.ethnicity_head.parameters(),  "lr": 2e-4},
            {"params": model.expression_head.parameters(), "lr": 2e-4},
        ],
        weight_decay = 1e-2,
    )
    scheduler_v2 = ReduceLROnPlateau(
        optimizer_v2, mode="min", factor=0.5, patience=4, min_lr=1e-7
    )

    model, hist2_train, hist2_val = train_model(
        model,
        dataloaders,
        criterion  = criterion,
        optimizer  = optimizer_v2,
        scheduler  = scheduler_v2,
        num_epochs = 40,
        patience   = 8,
        model_path = model_filename,
        device     = device,
        **AUX_WEIGHTS,
    )

else:
    # ---------------------------------------------------------
    # LOAD PRE-TRAINED MODEL
    # ---------------------------------------------------------
    print(f"Loading weights from {model_filename}...")
    model.load_state_dict(
        torch.load(model_filename, map_location=device, weights_only=True)
    )

    try:
        from PIL import Image
        import matplotlib.pyplot as plt
        img = Image.open("./ViTbaseline_train_history.png")
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    except FileNotFoundError:
        print("Training history image not found.")


>>> STAGE 2: Fine-Tuning Entire Model (Backbone Unfrozen)
Backbone status: Unfrozen

Epoch 1/25


Train Epoch 1: 100%|██████████| 64/64 [20:54<00:00, 19.60s/it, age_loss=0.0252, total_loss=0.363]


train Loss: 0.523982


Val Epoch 1: 100%|██████████| 24/24 [01:14<00:00,  3.10s/it, age_loss=0.0436, total_loss=1.99]


val Loss: 1.659763
Saving best model...

Epoch 2/25


...

val Loss: 1.852952

Epoch 9/25


Train Epoch 9: 100%|██████████| 64/64 [24:18<00:00, 22.79s/it, age_loss=0.0156, total_loss=0.175]


train Loss: 0.233848


Val Epoch 9: 100%|██████████| 24/24 [01:15<00:00,  3.15s/it, age_loss=0.0471, total_loss=1.78]


val Loss: 1.716601
Early stopping triggered.


# **Defining an auxiliary function to evaluate the model**
- Check Part I for the details.


In [17]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from zipfile import ZipFile
from torch.utils.data import DataLoader


def predict_and_evaluate(
    model,
    test_dataset,
    device,
    output_zip  = None,
    batch_size  = 32,
    age_norm    = 100.0,
):
    model.eval()

    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
    )

    # ── Accumulators ─────────────────────────────────────────
    all_age_preds   = []
    all_gen_preds   = []
    all_eth_preds   = []
    all_exp_preds   = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)

            out = model(images)

            all_age_preds.extend(
                out["age"].cpu().numpy())
            all_gen_preds.extend(
                out["gender"].argmax(dim=1).cpu().numpy())
            all_eth_preds.extend(
                out["ethnicity"].argmax(dim=1).cpu().numpy())
            all_exp_preds.extend(
                out["expression"].argmax(dim=1).cpu().numpy())

    # ── Build results dataframe ───────────────────────────────
    df = test_dataset.data_info.copy()

    df["predicted_age"]       = np.array(all_age_preds) * age_norm
    df["predicted_gender"]    = all_gen_preds
    df["predicted_ethnicity"] = all_eth_preds
    df["predicted_expression"]= all_exp_preds

    df["actual_age"]  = df["age"].astype(float)
    df["abs_error"]   = np.abs(df["predicted_age"] - df["actual_age"])

    # ── Age MAE ───────────────────────────────────────────────
    mae = df["abs_error"].mean()

    # ── Bias: std of per-group MAE ────────────────────────────
    def group_mae(col):
        stats = df.groupby(col)["abs_error"].agg(["mean", "count"])
        return stats["mean"].std(), stats["mean"].to_dict()

    gender_bias,  gender_detail  = group_mae("gender")
    eth_bias,     eth_detail     = group_mae("ethnicity")

    # Expression: exclude UTK rows (source == "utk" has no expression label)
    appa_df = df[df["source"] == "appa"]
    exp_bias, exp_detail = (
        group_mae.__func__(appa_df, "expression")   # reuse on subset
        if hasattr(group_mae, "__func__")
        else _group_mae_on(appa_df, "expression")
    )

    # Age bins
    df["age_bin"] = pd.cut(
        df["actual_age"],
        bins   = [0, 10, 20, 30, 40, 50, 60, 70, 100],
        labels = ["0-10","10-20","20-30","30-40",
                  "40-50","50-60","60-70","70+"]
    )
    age_bin_maes      = df.groupby("age_bin", observed=True)["abs_error"].mean()
    age_bias          = age_bin_maes.std()
    age_bin_detail    = age_bin_maes.to_dict()

    # Auxiliary head accuracy (where labels are valid)
    def head_acc(pred_col, label_col, subset=None):
        d = df if subset is None else df[subset]
        valid = d[label_col].notna()
        return (d.loc[valid, pred_col] == d.loc[valid, label_col]).mean()

    gen_acc = head_acc("predicted_gender",     "gender")
    eth_acc = head_acc("predicted_ethnicity",  "ethnicity")
    exp_acc = head_acc("predicted_expression", "expression",
                       subset=df["source"] == "appa")

    avg_bias = (gender_bias + eth_bias + exp_bias + age_bias) / 4

    # ── Report ────────────────────────────────────────────────
    sep = "=" * 36
    print(f"\n{sep}")
    print(f"  EVALUATION REPORT")
    print(sep)
    print(f"  Global MAE          : {mae:.4f} yrs  ({mae * age_norm:.2f} raw)")
    print(f"\n  Auxiliary Accuracy")
    print(f"    Gender            : {gen_acc*100:.1f}%")
    print(f"    Ethnicity         : {eth_acc*100:.1f}%")
    print(f"    Expression (APPA) : {exp_acc*100:.1f}%")
    print(f"\n  Bias (std of group MAE)")
    print(f"    Gender            : {gender_bias:.4f}")
    for k, v in gender_detail.items():
        print(f"      {k:<16}: {v:.4f}")
    print(f"    Ethnicity         : {eth_bias:.4f}")
    for k, v in eth_detail.items():
        print(f"      {k:<16}: {v:.4f}")
    print(f"    Expression        : {exp_bias:.4f}")
    for k, v in exp_detail.items():
        print(f"      {k:<16}: {v:.4f}")
    print(f"    Age bins          : {age_bias:.4f}")
    for k, v in age_bin_detail.items():
        print(f"      {k:<16}: {v:.4f}")
    print(f"\n  Avg Bias            : {avg_bias:.4f}")
    print(sep)

    # ── Optional zip export ───────────────────────────────────
    if output_zip:
        pred_path = "predictions.csv"
        df[["predicted_age"]].to_csv(pred_path, index=False, header=False)
        with ZipFile(output_zip, "w") as z:
            z.write(pred_path)
        print(f"  Predictions saved → {output_zip}")

    metrics = {
        "mae":          mae,
        "gender_bias":  gender_bias,
        "eth_bias":     eth_bias,
        "exp_bias":     exp_bias,
        "age_bias":     age_bias,
        "avg_bias":     avg_bias,
        "gender_acc":   gen_acc,
        "eth_acc":      eth_acc,
        "exp_acc":      exp_acc,
        "gender_detail": gender_detail,
        "eth_detail":    eth_detail,
        "exp_detail":    exp_detail,
        "age_bin_detail":age_bin_detail,
    }

    return df, metrics


# ── helper (needed because group_mae is a closure) ───────────
def _group_mae_on(df, col):
    stats = df.groupby(col)["abs_error"].agg(["mean", "count"])
    return stats["mean"].std(), stats["mean"].to_dict()

In [18]:
model_filename = "./best_multitask_model_with_augmentation.pth"
model.load_state_dict(torch.load(model_filename, map_location=device))
import os
print(os.path.exists(model_filename), os.path.getsize(model_filename))

True 345137182


# **Loading the Saved Model and Making Predictions on the Validation Set**


In [20]:
model_filename = "./best_multitask_model.pth"

# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_valid,output_zip=None, batch_size=32)

Predicting: 100%|██████████| 47/47 [01:14<00:00,  1.59s/it]


METRICS REPORT
Global MAE: 4.0673
Gender Bias (std): 0.0995
Ethnicity Bias (std): 0.2831
Expression Bias (std): 0.2938
Age Bias (std): 2.0751
--- OVERALL AVG BIAS: 0.6879 ---




---



# **Generating the submission fie (on the test set) for our challenge**

- Loading the Saved Model and Making Predictions on the Test Set

- The following cells are generating predictions (and evaluating them) on the **Test set** so that we can create our submission file to be uploaded to our age estimation challenge.

- **Do not evaluate your model on the Test set when defining your model, training strategy, or hyperparameters.** For this, use the Validation set.

In [21]:
# Create dataset and dataloader (test set):
dataset_test = AgeEstimationDataset("content/appa-real-dataset_v2/test_data", "content/appa-real-dataset_v2/labels_metadata_test.csv", transforms=base_transforms)
print(f"Total number of test samples: {len(dataset_test)}")

Total number of test samples: 1978


In [22]:
# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_test, "predictions_multihead_task2.zip")

Predicting: 100%|██████████| 62/62 [01:53<00:00,  1.83s/it]


METRICS REPORT
Global MAE: 4.7404
Gender Bias (std): 0.0694
Ethnicity Bias (std): 0.2527
Expression Bias (std): 0.1421
Age Bias (std): 2.9001
--- OVERALL AVG BIAS: 0.8411 ---
